# Orchestration Notebook

This notebook demonstrates the **Orchestrator** component, which coordinates the multi-agent pipeline.

## Pipeline Architecture

```
Designer (Always) → [Validator] → [Optimizer ⟷ Validator Loop] → Final Validator (Always) → [Educational]
```

> **Note:** Educational Agent runs at the END (after Final Validator) to conserve system resources.

## Features Demonstrated
1. **Component Initialization** - Setting up all agents and orchestrator
2. **Pipeline Status** - Viewing configuration and enabled stages
3. **Simple Generation** - Basic code generation
4. **Full Pipeline** - All stages including optimization loop and educational
5. **Workflow Manager** - Custom workflow definition and execution

---

# Part 1: Setup & Configuration

In [1]:
import sys
import os
from pathlib import Path

# Add project root to path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.cirq_rag_code_assistant.config import get_config, setup_logging
from src.rag.knowledge_base import KnowledgeBase
from src.rag.retriever import Retriever
from src.rag.generator import Generator
from src.agents.designer import DesignerAgent
from src.agents.optimizer import OptimizerAgent
from src.agents.validator import ValidatorAgent
from src.agents.educational import EducationalAgent
from src.orchestration.orchestrator import Orchestrator
from src.orchestration.workflow_manager import WorkflowManager

# Setup logging
setup_logging()

# Knowledge base path
KNOWLEDGE_BASE_DIR = project_root / "data" / "knowledge_base"
print(f"Knowledge Base Directory: {KNOWLEDGE_BASE_DIR}")
print("✅ Imports completed successfully.")

2025-12-07 15:51:19 | INFO     | src.cirq_rag_code_assistant.config.logging:setup_all:138 | Logging configuration completed


Knowledge Base Directory: D:\University\Uni\Semester 7\Generative AI\Project\Cirq-RAG-Code-Assistant\data\knowledge_base
✅ Imports completed successfully.


## 1.1 Display Configuration

View the orchestration-related configuration settings.

In [2]:
config = get_config()

# General agent settings
general = config.get("agents", {}).get("general", {})
print("📋 General Agent Configuration:")
print(f"   max_retries: {general.get('max_retries', 3)}")
print(f"   timeout_seconds: {general.get('timeout_seconds', 300)}")
print(f"   parallel_execution: {general.get('parallel_execution', True)}")
print()

# Agent enabled states
print("🤖 Agent Enabled Status:")
agents_config = config.get("agents", {})
for agent_name in ["designer", "optimizer", "validator", "educational"]:
    enabled = agents_config.get(agent_name, {}).get("enabled", True)
    status = "✅ Enabled" if enabled else "❌ Disabled"
    print(f"   {agent_name.capitalize()}: {status}")
print()

# Optimizer settings
optimizer_config = agents_config.get("optimizer", {})
print("⚡ Optimizer Configuration:")
print(f"   level: {optimizer_config.get('level', 'balanced')}")
print(f"   use_rl: {optimizer_config.get('use_rl', False)}")
print(f"   rl_iterations: {optimizer_config.get('rl_iterations', 10)}")

2025-12-07 15:51:19 | INFO     | config.config_loader:load:93 | ✅ Loaded configuration from D:\University\Uni\Semester 7\Generative AI\Project\Cirq-RAG-Code-Assistant\config\config.json


📋 General Agent Configuration:
   max_retries: 3
   timeout_seconds: 300
   parallel_execution: True

🤖 Agent Enabled Status:
   Designer: ✅ Enabled
   Optimizer: ✅ Enabled
   Validator: ✅ Enabled
   Educational: ✅ Enabled

⚡ Optimizer Configuration:
   level: balanced
   use_rl: False
   rl_iterations: 10


---

# Part 2: Component Initialization

## 2.1 Initialize RAG Components

In [3]:
# Initialize Knowledge Base and RAG
print("📚 Initializing RAG components...")

# Create KnowledgeBase with path and load entries
kb = KnowledgeBase(knowledge_base_path=KNOWLEDGE_BASE_DIR)
num_loaded = kb.load_from_directory()
print(f"   Loaded {num_loaded} entries from knowledge base")

# Index entries if needed
if num_loaded > 0:
    kb.index_entries()
    print("   Entries indexed in vector store")

# Initialize Retriever and Generator
retriever = Retriever(knowledge_base=kb)
generator = Generator(retriever=retriever)

# Get knowledge base stats
kb_stats = kb.get_stats()
print(f"   Knowledge Base: {kb_stats.get('total_entries', 0)} total entries")
print("✅ RAG components initialized.")

2025-12-07 15:51:19 | INFO     | src.rag.embeddings:__init__:97 | Loading embedding model: BAAI/bge-base-en-v1.5
2025-12-07 15:51:19 | INFO     | src.rag.embeddings:__init__:98 | Using device: cpu


📚 Initializing RAG components...
2025-12-07 15:51:19,628 - sentence_transformers.SentenceTransformer - INFO - SentenceTransformer.py:227 - Load pretrained SentenceTransformer: BAAI/bge-base-en-v1.5


2025-12-07 15:51:23 | INFO     | src.rag.embeddings:__init__:106 | ✅ Embedding model loaded successfully
2025-12-07 15:51:23 | INFO     | src.rag.embeddings:__init__:113 | Embedding dimension: 768
2025-12-07 15:51:23 | INFO     | src.rag.vector_store:_init_faiss:139 | Initialized FAISS index
2025-12-07 15:51:23 | INFO     | src.rag.vector_store:__init__:120 | Initialized VectorStore with faiss backend
2025-12-07 15:51:23 | INFO     | src.rag.vector_store:__init__:121 | Embedding dimension: 768
2025-12-07 15:51:23 | INFO     | src.rag.knowledge_base:__init__:100 | Initialized KnowledgeBase with 0 entries
2025-12-07 15:51:23 | INFO     | src.rag.knowledge_base:load_from_jsonl:116 | Loading knowledge base from D:\University\Uni\Semester 7\Generative AI\Project\Cirq-RAG-Code-Assistant\data\knowledge_base\curated_designer_examples.jsonl
2025-12-07 15:51:23 | INFO     | src.rag.knowledge_base:load_from_jsonl:129 | Loaded 107 entries from D:\University\Uni\Semester 7\Generative AI\Project\Cir

   Loaded 140 entries from knowledge base


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

2025-12-07 15:51:43 | INFO     | src.rag.knowledge_base:index_entries:248 | ✅ Indexed 140 entries
2025-12-07 15:51:43 | INFO     | src.rag.retriever:__init__:170 | Initialized Retriever with top_k=5, threshold=0.7, topic_boost=0.15, hybrid_scoring=True
2025-12-07 15:51:43 | INFO     | src.rag.generator:__init__:170 | Initialized Generator with ollama/cirq-designer-agent


   Entries indexed in vector store
   Knowledge Base: 140 total entries
✅ RAG components initialized.


## 2.2 Initialize All Agents

In [4]:
print("🤖 Initializing agents...")

# Designer Agent (always runs first)
designer = DesignerAgent(retriever=retriever, generator=generator)
print("   ✅ Designer Agent")

# Optimizer Agent with RAG for optimization guidance
optimizer = OptimizerAgent(retriever=retriever)
print("   ✅ Optimizer Agent")

# Validator Agent (using remote mode with QCanvas)
try:
    validator = ValidatorAgent(retriever=retriever)  # Mode is read from config.json
    print(f"   ✅ Validator Agent (mode={validator.mode})")
except Exception as e:
    validator = ValidatorAgent(retriever=retriever)  # Mode is read from config.json
    print(f"   ⚠️ Validator Agent (fallback to local mode)")

# Educational Agent (runs at end to conserve resources)
educational = EducationalAgent(retriever=retriever)
print("   ✅ Educational Agent (runs at end of pipeline)")

print()
print("✅ All agents initialized!")


2025-12-07 15:51:43 | INFO     | src.agents.base_agent:__init__:78 | Initialized DesignerAgent agent
2025-12-07 15:51:43 | INFO     | src.agents.base_agent:__init__:78 | Initialized OptimizerAgent agent
2025-12-07 15:51:43 | INFO     | src.rag.generator:__init__:170 | Initialized Generator with ollama/cirq-optimizer-agent
2025-12-07 15:51:43 | INFO     | src.agents.optimizer:__init__:74 | OptimizerAgent initialized (RAG: enabled)
2025-12-07 15:51:43 | INFO     | src.agents.base_agent:__init__:78 | Initialized ValidatorAgent agent
2025-12-07 15:51:43 | INFO     | src.tools.simulator:__init__:82 | Initialized simulator simulator
2025-12-07 15:51:43 | INFO     | src.agents.validator:__init__:94 | ValidatorAgent initialized in 'local' mode (RAG: enabled)
2025-12-07 15:51:43 | INFO     | src.agents.base_agent:__init__:78 | Initialized EducationalAgent agent
2025-12-07 15:51:43 | INFO     | src.agents.educational:__init__:239 | EducationalAgent using config from: agents.educational.model
202

🤖 Initializing agents...
   ✅ Designer Agent
   ✅ Optimizer Agent
   ✅ Validator Agent (mode=local)
   ✅ Educational Agent (runs at end of pipeline)

✅ All agents initialized!


## 2.3 Initialize Orchestrator

In [5]:
# Initialize the Orchestrator with all agents
orchestrator = Orchestrator(
    designer=designer,
    optimizer=optimizer,
    validator=validator,
    educational=educational
)

print("✅ Orchestrator initialized!")
print()

# Display pipeline status
status = orchestrator.get_pipeline_status()
print("📊 Pipeline Status:")
for component, info in status.items():
    if component == "config":
        print(f"\n⚙️ Config:")
        for key, value in info.items():
            print(f"   {key}: {value}")
    else:
        avail = "✅" if info.get("available") else "❌"
        enabled = "Enabled" if info.get("enabled") else "Disabled"
        print(f"   {component.capitalize()}: {avail} Available, {enabled}")

2025-12-07 15:51:43 | INFO     | src.orchestration.orchestrator:__init__:80 | Initialized Orchestrator (max_retries=3, timeout=300s)


✅ Orchestrator initialized!

📊 Pipeline Status:
   Designer: ✅ Available, Enabled
   Validator: ✅ Available, Enabled
   Optimizer: ✅ Available, Enabled
   Educational: ✅ Available, Enabled

⚙️ Config:
   max_retries: 3
   timeout_seconds: 300
   parallel_execution: True


---

# Part 3: Simple Code Generation

Run the pipeline with just the Designer (no validation/optimization).

In [6]:
simple_query = "Create a simple Bell state circuit with 2 qubits"

print(f"🔍 Query: {simple_query}")
print()

# Run with minimal pipeline (Designer only)
result = orchestrator.generate_code(
    query=simple_query,
    algorithm="bell_state",
    optimize=False,  # Skip optimizer
    validate=False,  # Skip validator
    explain=False,   # Skip educational
)

print(f"Success: {result['success']}")
print(f"Stages Run: {result['stages']}")
print()

if result['success']:
    print("📄 Generated Code:")
    print("-" * 50)
    print(result['code'])
    print("-" * 50)
else:
    print(f"❌ Errors: {result.get('errors')}")

2025-12-07 15:51:43 | INFO     | src.orchestration.orchestrator:generate_code:127 | 🎨 [Stage 1/4] Designer Agent generating code...
2025-12-07 15:51:43 | INFO     | src.rag.generator:generate:193 | Retrieving context for query: Create a simple Bell state circuit with 2 qubits...
2025-12-07 15:51:43 | INFO     | src.rag.generator:generate:218 | Generating code using ollama/cirq-designer-agent


🔍 Query: Create a simple Bell state circuit with 2 qubits



2025-12-07 15:51:59 | INFO     | src.rag.generator:generate:273 | ✅ Code generation completed
2025-12-07 15:51:59 | INFO     | src.orchestration.orchestrator:generate_code:141 | ✅ Designer generated code (552 chars)
2025-12-07 15:51:59 | INFO     | src.orchestration.orchestrator:generate_code:162 | ⏭️ [Stage 2/4] Validator skipped (disabled or not available)
2025-12-07 15:51:59 | INFO     | src.orchestration.orchestrator:generate_code:213 | ⏭️ [Stage 3/4] Optimizer skipped (disabled or not available)
2025-12-07 15:51:59 | INFO     | src.orchestration.orchestrator:generate_code:219 | 🔍 [Stage 4/4] Final Validator ensuring quality...
2025-12-07 15:51:59 | ERROR    | src.tools.simulator:simulate:158 | Simulation error: Circuit has no measurements to sample.
2025-12-07 15:51:59 | WARNING  | src.agents.validator:_execute_local:438 | Simulation failed: Circuit has no measurements to sample.
2025-12-07 15:51:59 | INFO     | src.agents.validator:_execute_local:496 | Running LLM analysis for co

Success: True
Stages Run: ['designer', 'final_validator']

📄 Generated Code:
--------------------------------------------------
import cirq

def create_bell_state():
    # Create two qubits
    qubit1 = cirq.GridQubit(0, 0)
    qubit2 = cirq.GridQubit(0, 1)
    
    # Create a circuit with Hadamard gate on the first qubit and CNOT gate between the two qubits
    circuit = cirq.Circuit(
        cirq.H(qubit1),  # Apply Hadamard gate to put qubit1 in superposition
        cirq.CNOT(qubit1, qubit2)  # Apply CNOT gate with qubit1 as control and qubit2 as target
    )
    
    return circuit

# Print the Bell state circuit
bell_circuit = create_bell_state()
print(bell_circuit)
--------------------------------------------------


---

# Part 4: Full Pipeline Demo

Run the complete pipeline:
```
Designer → Validator → Optimizer (with loop) → Final Validator → Educational
```

> **Note:** Educational Agent runs at the END to avoid resource conflicts.

In [7]:
full_query = "Create a Grover's algorithm circuit for searching a 2-qubit system"

print(f"🔍 Query: {full_query}")
print(f"🔄 Running full pipeline...")
print()

# Run full pipeline
result = orchestrator.generate_code(
    query=full_query,
    algorithm="grover",
    optimize=True,
    validate=True,
    explain=True,  # Educational runs at the END
    max_optimization_loops=2,
)

print()
print("=" * 50)
print("📊 RESULTS")
print("=" * 50)
print(f"Success: {result['success']}")
print(f"Stages Completed: {result['stages']}")
print()

2025-12-07 15:52:17 | INFO     | src.orchestration.orchestrator:generate_code:127 | 🎨 [Stage 1/4] Designer Agent generating code...
2025-12-07 15:52:17 | INFO     | src.rag.generator:generate:193 | Retrieving context for query: Create a Grover's algorithm circuit for searching ...
2025-12-07 15:52:18 | INFO     | src.rag.generator:generate:218 | Generating code using ollama/cirq-designer-agent


🔍 Query: Create a Grover's algorithm circuit for searching a 2-qubit system
🔄 Running full pipeline...



2025-12-07 15:52:38 | INFO     | src.rag.generator:generate:273 | ✅ Code generation completed
2025-12-07 15:52:38 | INFO     | src.orchestration.orchestrator:generate_code:141 | ✅ Designer generated code (1204 chars)
2025-12-07 15:52:38 | INFO     | src.orchestration.orchestrator:generate_code:145 | ✅ [Stage 2/4] Validator Agent checking code...
2025-12-07 15:52:38 | INFO     | src.tools.simulator:simulate:147 | Simulation completed. Histogram: {'01': 276, '00': 251, '11': 243, '10': 254}
2025-12-07 15:52:38 | INFO     | src.orchestration.orchestrator:generate_code:166 | ⚡ [Stage 3/4] Optimizer Agent improving code...
2025-12-07 15:52:38 | INFO     | src.orchestration.orchestrator:generate_code:172 |    Optimization iteration 1/2
2025-12-07 15:52:38 | INFO     | src.agents.optimizer:execute:280 | Retrieved 3 optimization references from RAG
2025-12-07 15:52:38 | INFO     | src.tools.simulator:simulate:147 | Simulation completed. Histogram: {'00': 237, '11': 257, '10': 275, '01': 255}
2


📊 RESULTS
Success: True
Stages Completed: ['designer', 'validator', 'optimizer', 'final_validator', 'educational']



In [8]:
# Display final code
if result['success']:
    final_code = result.get('optimized_code') or result.get('code')
    print("📄 Final Code:")
    print("-" * 50)
    print(final_code)
    print("-" * 50)
    print()
    
    # Optimization metrics
    if result.get('optimization_metrics'):
        print("⚡ Optimization Metrics:")
        for key, value in result['optimization_metrics'].items():
            print(f"   {key}: {value}")
        print()
    
    # Validation results
    if result.get('final_validation'):
        fv = result['final_validation']
        status = "✅ PASSED" if fv.get('validation_passed') else "⚠️ WARNINGS"
        print(f"🔍 Final Validation: {status}")
        print()
    
    # Educational content (runs at the end)
    if result.get('explanation'):
        print("📚 Educational Explanation (generated at end of pipeline):")
        print("-" * 50)
        explanation = result['explanation']
        if isinstance(explanation, dict) and explanation.get('markdown'):
            print(explanation['markdown'] + "...")
        else:
            print(str(explanation) + "...")
        print("-" * 50)
else:
    print(f"❌ Pipeline failed with errors:")
    for error in result.get('errors', []):
        print(f"   - {error}")

📄 Final Code:
--------------------------------------------------
import cirq

q0 = cirq.GridQubit(0, 0)
q1 = cirq.GridQubit(0, 1)
qubits = [q0, q1]

circuit = cirq.Circuit()
circuit.append(cirq.PhasedXZGate(axis_phase_exponent=-0.5, x_exponent=0.5, z_exponent=-1.0).on(qubits[0]))
circuit.append(cirq.PhasedXZGate(axis_phase_exponent=-0.5, x_exponent=0.5, z_exponent=-1.0).on(qubits[1]))
circuit.append(cirq.CNOT(qubits[0], qubits[1]))
circuit.append(cirq.PhasedXZGate(axis_phase_exponent=-0.5, x_exponent=0.0, z_exponent=-1.0).on(qubits[1]))
circuit.append(cirq.CNOT(qubits[0], qubits[1]))
circuit.append(cirq.PhasedXZGate(axis_phase_exponent=-0.5, x_exponent=0.5, z_exponent=-1.0).on(qubits[0]))
circuit.append(cirq.PhasedXZGate(axis_phase_exponent=-0.5, x_exponent=0.5, z_exponent=-1.0).on(qubits[1]))
circuit.append(cirq.CZ(qubits[1], qubits[0]))
circuit.append(cirq.PhasedXZGate(axis_phase_exponent=-0.5, x_exponent=0.5, z_exponent=-1.0).on(qubits[0]))
circuit.append(cirq.PhasedXZGate(axis_phas

---

# Part 5: Workflow Manager Demo

The WorkflowManager allows defining custom multi-step workflows.

In [9]:
# Initialize Workflow Manager
wf_manager = WorkflowManager()
print("✅ WorkflowManager initialized")
print(f"   Default timeout: {wf_manager.default_timeout}s")
print(f"   Max retries: {wf_manager.max_retries}")

2025-12-07 15:53:15 | INFO     | src.orchestration.workflow_manager:__init__:80 | Initialized WorkflowManager (timeout=300s, retries=3)


✅ WorkflowManager initialized
   Default timeout: 300s
   Max retries: 3


In [10]:
# Define a custom workflow
def step_generate(state):
    """Generate code based on query in state."""
    query = state.get("query", "Create a Bell state")
    result = designer.run({"query": query})
    return {"code": result.get("code"), "design_success": result.get("success")}

def step_validate(state):
    """Validate the generated code."""
    code = state.get("code")
    if not code:
        return {"validation_passed": False, "error": "No code to validate"}
    result = validator.execute({"code": code})
    return {"validation_passed": result.get("validation_passed", False)}

def step_report(state):
    """Generate a summary report."""
    return {
        "report": f"Design: {'✅' if state.get('design_success') else '❌'}, "
                  f"Validation: {'✅' if state.get('validation_passed') else '❌'}"
    }

# Define the workflow
wf_manager.define_workflow(
    name="simple_generation",
    description="Generate and validate quantum code",
    steps=[
        {"name": "generate", "function": step_generate, "required": True},
        {"name": "validate", "function": step_validate, "required": False},  # Optional
        {"name": "report", "function": step_report, "required": True},
    ]
)

print("✅ Workflow 'simple_generation' defined")
print()

# List all workflows
print("📋 Available Workflows:")
for wf in wf_manager.list_workflows():
    print(f"   - {wf['name']}: {wf['description']} ({wf['steps']} steps)")

2025-12-07 15:53:15 | INFO     | src.orchestration.workflow_manager:define_workflow:117 | Defined workflow: 'simple_generation' with 3 steps


✅ Workflow 'simple_generation' defined

📋 Available Workflows:
   - simple_generation: Generate and validate quantum code (3 steps)


In [11]:
# Execute the workflow
print("▶️ Executing workflow 'simple_generation'...")
print()

wf_result = wf_manager.execute_workflow(
    name="simple_generation",
    initial_state={"query": "Create a quantum teleportation circuit"}
)

print()
print("=" * 50)
print("📊 WORKFLOW RESULTS")
print("=" * 50)
print(f"Success: {wf_result['success']}")
print(f"Steps Completed: {wf_result['steps_completed']}/{wf_result['steps_total']}")
print()

# Show step results
print("📋 Step Results:")
for step_res in wf_result['step_results']:
    status = "✅" if step_res['success'] else "❌"
    print(f"   {status} {step_res['step']}")

print()
print(f"📝 Final Report: {wf_result['state'].get('report', 'N/A')}")

2025-12-07 15:53:15 | INFO     | src.orchestration.workflow_manager:execute_workflow:160 | ▶️ Starting workflow: 'simple_generation' (3 steps)
2025-12-07 15:53:15 | INFO     | src.orchestration.workflow_manager:execute_workflow:167 |   [1/3] Executing step: generate
2025-12-07 15:53:15 | INFO     | src.rag.generator:generate:193 | Retrieving context for query: Create a quantum teleportation circuit...
2025-12-07 15:53:15 | INFO     | src.rag.generator:generate:218 | Generating code using ollama/cirq-designer-agent


▶️ Executing workflow 'simple_generation'...



2025-12-07 15:53:34 | INFO     | src.rag.generator:generate:273 | ✅ Code generation completed
2025-12-07 15:53:34 | INFO     | src.agents.base_agent:run:130 | DesignerAgent retrying (attempt 2/3)
2025-12-07 15:53:34 | INFO     | src.rag.generator:generate:193 | Retrieving context for query: Create a quantum teleportation circuit...
2025-12-07 15:53:34 | INFO     | src.rag.generator:generate:218 | Generating code using ollama/cirq-designer-agent
2025-12-07 15:53:44 | INFO     | src.rag.generator:generate:273 | ✅ Code generation completed
2025-12-07 15:53:44 | INFO     | src.agents.base_agent:run:130 | DesignerAgent retrying (attempt 3/3)
2025-12-07 15:53:44 | INFO     | src.rag.generator:generate:193 | Retrieving context for query: Create a quantum teleportation circuit...
2025-12-07 15:53:44 | INFO     | src.rag.generator:generate:218 | Generating code using ollama/cirq-designer-agent
2025-12-07 15:53:55 | INFO     | src.rag.generator:generate:273 | ✅ Code generation completed
2025-12-


📊 WORKFLOW RESULTS
Success: True
Steps Completed: 3/3

📋 Step Results:
   ✅ generate
   ✅ validate
   ✅ report

📝 Final Report: Design: ❌, Validation: ❌


---

# Summary

This notebook demonstrated:

1. **Configuration** - Loading orchestration settings from `config.json`
2. **Component Initialization** - Setting up RAG, agents, and orchestrator
3. **Pipeline Status** - Viewing enabled/disabled stages
4. **Simple Generation** - Running minimal pipeline (Designer only)
5. **Full Pipeline** - All stages with optimization loop
6. **Workflow Manager** - Custom workflow definition and execution

## Key Features

- **Config-driven**: All settings loaded from `config/config.json`
- **Retry logic**: Automatic retries based on `agents.general.max_retries`
- **Optimization loop**: Optimizer can request re-validation
- **Sequential execution**: Educational Agent runs at the END to conserve resources
- **Flexible workflows**: WorkflowManager for custom pipelines